# Imports

In [1]:
import ast
import yaml
import json
import requests
from collections import defaultdict
from lxml import etree
from elasticsearch import Elasticsearch
from bs4 import BeautifulSoup

In [2]:
annotations = defaultdict(list)
for filename in  ["darwin_tree_of_life", "erga_bge", "erga_pilot"]:
    with open(f'/Users/alexey/ebi_projects/projects.ensembl.org/_data/{filename}/species.yaml', 'r') as yaml_file:
        yaml_data = yaml.safe_load(yaml_file)
        for record in yaml_data:
            annotation = dict()
            annotation['species'] = record['species']
            annotation['accession'] = record['accession']
            print(f"Working on {annotation['accession']} from {filename}")
            acc_response = requests.get(f"https://www.ebi.ac.uk/ena/browser/api/xml/{annotation['accession']}")
            try:
                root = etree.fromstring(acc_response.content)
            except etree.XMLSyntaxError:
                print(annotation['accession'])
                continue
            try:
                tax_id = root.find("ASSEMBLY").find("TAXON").find("TAXON_ID").text
            except AttributeError:
                if annotation['accession'] == "GCF_902459465.1":
                    tax_id = "7604"
                elif annotation['accession'] == "GCF_902652985.1":
                    tax_id = "6579"
            annotation['tax_id'] = tax_id
            annotation['annotation'] = {'GTF': record['annotation_gtf'], "GFF3": record['annotation_gff3']}
            annotation['proteins'] = {'FASTA': record['proteins']}
            annotation['transcripts'] = {'FASTA': record['transcripts']}
            annotation['softmasked_genome'] = {'FASTA': record['softmasked_genome']}
            try:
                annotation['repeat_library'] = {'FASTA': record['repeat_library']}
            except KeyError:
                annotation['repeat_library'] = None
            annotation['other_data'] = {'ftp_dumps': record['ftp_dumps']}
            try:
                annotation['view_in_browser'] = record['rapid_link']
            except KeyError:
                annotation['view_in_browser'] = None
            annotations[annotation["tax_id"]].append(annotation)
len(annotations)

Working on GCA_963993115.1 from darwin_tree_of_life
Working on GCA_964187985.1 from darwin_tree_of_life
Working on GCA_905340225.1 from darwin_tree_of_life
Working on GCA_946251915.1 from darwin_tree_of_life
Working on GCA_930367205.1 from darwin_tree_of_life
Working on GCA_929443795.2 from darwin_tree_of_life
Working on GCA_943193645.1 from darwin_tree_of_life
Working on GCA_954870605.1 from darwin_tree_of_life
Working on GCA_947623365.1 from darwin_tree_of_life
Working on GCA_948252455.1 from darwin_tree_of_life
Working on GCA_927399475.2 from darwin_tree_of_life
Working on GCA_963966035.1 from darwin_tree_of_life
Working on GCA_946894065.1 from darwin_tree_of_life
Working on GCA_923062465.1 from darwin_tree_of_life
Working on GCA_963555685.1 from darwin_tree_of_life
Working on GCA_963576875.1 from darwin_tree_of_life
Working on GCA_943193695.1 from darwin_tree_of_life
Working on GCA_947359355.1 from darwin_tree_of_life
Working on GCA_910591435.1 from darwin_tree_of_life
Working on G

KeyboardInterrupt: 

In [3]:
root = etree.fromstring(acc_response.content)

# Constants

In [3]:
ES_HOST = "https://prj-ext-dev-dtol-gcp-dr-349815.es.europe-west2.gcp.elastic-cloud.com"
ES_USERNAME = "elastic"
ES_PASSWORD = "zLNNQVaYJvFPhLFMbwiB1uLt"

In [4]:
DATA_PORTAL_AGGREGATIONS_DTOL = ["assemblies_status", "annotation_complete"]

In [9]:
def update_summary_index_dtol():
    es = Elasticsearch([ES_HOST], http_auth=("elastic", ES_PASSWORD))
    body = dict()
    body["aggs"] = dict()
    for aggregation_field in DATA_PORTAL_AGGREGATIONS_DTOL:
        body["aggs"][aggregation_field] = {
            "terms": {"field": aggregation_field, "size": 20}
        }
    body["aggs"]["genome_notes"] = {
        "nested": {"path": "genome_notes"},
        "aggs": {
            "genome_count": {
                "reverse_nested": {},  # get to the parent document level
                "aggs": {
                    "distinct_docs": {"cardinality": {"field": "organism.keyword"}}
                },
            }
        },
    }
    results = es.search(index="2026-02-09_data_portal", body=body)
    summary = dict()
    names_mapping = {
        "assemblies_status": "Assemblies - Submitted",
        "annotation_complete": "Annotation Complete",
        "genome_notes": "Genome Notes",
    }
    for key, aggs in results["aggregations"].items():
        try:
            for bucket in aggs["buckets"]:
                if bucket["key"] == "Done":
                    if key in names_mapping:
                        summary[names_mapping[key]] = bucket["doc_count"]
        except KeyError:
            summary["Genome Notes"] = aggs["genome_count"]["distinct_docs"]["value"]
    es.index(index="summary", body=summary, id="summary")

In [10]:
update_summary_index_dtol()

/var/folders/71/4c_77lrn65x46bxkhf0k6bj80000gp/T/ipykernel_85426/2634982774.py:20: DeprecationWarning: The 'body' parameter is deprecated for the 'search' API and will be removed in a future version. Instead use API parameters directly. See https://github.com/elastic/elasticsearch-py/issues/1698 for more information
  results = es.search(index="2026-02-09_data_portal", body=body)
/var/folders/71/4c_77lrn65x46bxkhf0k6bj80000gp/T/ipykernel_85426/2634982774.py:35: DeprecationWarning: The 'body' parameter is deprecated for the 'index' API and will be removed in a future version. Instead use the 'document' parameter. See https://github.com/elastic/elasticsearch-py/issues/1698 for more information
  es.index(index="summary", body=summary, id="summary")


# Importing Data Portal Metadata

In [3]:
data = list()
with open("/Users/alexey/dtol_results.txt", "r") as f:
    for line in f:
        line = line.rstrip()
        data.append(ast.literal_eval(line))
len(data)

6631

In [4]:
# To remove duplicated records
for record in data:
    visited_ids = {}
    new_records = {}
    ranks = {
        "Submitted to BioSamples": 1,
        "Raw Data - Submitted": 2,
        "Assemblies - Submitted": 3
    }
    for sample in record["records"]:
        if sample["accession"] not in visited_ids:
            visited_ids[sample["accession"]] = ranks[sample["trackingSystem"]]
            new_records[sample["accession"]] = sample
        else:
            if ranks[sample["trackingSystem"]] > visited_ids[sample["accession"]]:
                visited_ids[sample["accession"]] = ranks[sample["trackingSystem"]]
                new_records[sample["accession"]] = sample
    record["records"] = list(new_records.values())

In [5]:
# To remove duplicated runs
for record in data:
    visited_ids = set()
    new_runs = list()
    for exp in record["experiment"]:
        if exp["run_accession"] not in visited_ids:
            visited_ids.add(exp["run_accession"])
            new_runs.append(exp)
    record["experiment"] = new_runs

In [6]:
# To remove duplicated assemblies
for record in data:
    visited_ids = set()
    new_assemblies = list()
    for assmbl in record["assemblies"]:
        if assmbl["accession"] not in visited_ids:
            visited_ids.add(assmbl["accession"])
            new_assemblies.append(assmbl)
    record["assemblies"] = new_assemblies

In [7]:
# To remove duplicated symbionts assemblies
for record in data:
    visited_ids = set()
    new_assemblies = list()
    for assmbl in record["symbionts_assemblies"]:
        if assmbl["accession"] not in visited_ids:
            visited_ids.add(assmbl["accession"])
            new_assemblies.append(assmbl)
    record["symbionts_assemblies"] = new_assemblies

In [8]:
# To remove duplicated metagenomes assemblies
for record in data:
    visited_ids = set()
    new_assemblies = list()
    for assmbl in record["metagenomes_assemblies"]:
        if assmbl["accession"] not in visited_ids:
            visited_ids.add(assmbl["accession"])
            new_assemblies.append(assmbl)
    record["metagenomes_assemblies"] = new_assemblies

In [9]:
es_data = list()
for record in data:
    es_data.append({"index": {"_index": "2025-01-27_data_portal", "_id": record['organism']}})
    if record['organism'] == 'Ochlodes sylvanus':
        record['tax_id'] = 3126489
    if 'goat_info' in record and record['goat_info'] is not None and 'results' in record['goat_info'] and len(record['goat_info']['results']) != 0:
        record['goat_info'] = record['goat_info']['results'][0]['_source']
    es_data.append(record)
es = Elasticsearch([ES_HOST], http_auth=(ES_USERNAME, ES_PASSWORD))
for i in range(0, len(es_data), 1000):
    print(f"Working on {i}: {i+1000}")
    _ = es.bulk(body=es_data[i:i+1000])

Working on 0: 1000
Working on 1000: 2000
Working on 2000: 3000
Working on 3000: 4000
Working on 4000: 5000
Working on 5000: 6000
Working on 6000: 7000
Working on 7000: 8000
Working on 8000: 9000
Working on 9000: 10000
Working on 10000: 11000
Working on 11000: 12000
Working on 12000: 13000
Working on 13000: 14000


# Importing statuses

In [3]:
es = Elasticsearch([ES_HOST], http_auth=(ES_USERNAME, ES_PASSWORD))
def get_samples(index_name, es):
    samples = dict()
    data = es.search(index=index_name, size=1000)
    offset = 0
    while len(data['hits']['hits']) > 0:
        for sample in data['hits']['hits']:
            samples[sample['_id']] = sample['_source']
        offset += 1000
        data = es.search(index=index_name, size=1000, from_=offset)
    return samples

samples = get_samples("organisms_test", es)

In [9]:
names = set()
for biosample_id, record in samples.items():
    names.add(record["organism"]["text"])
print(len(names))

8653


In [26]:
data_portal = get_samples("data_portal", es)
images_available = set()
for organism_name, record in data_portal.items():
    if record["images_available"] is True:
        images_available.add(organism_name)
print(len(images_available))

2225


In [10]:
list(images_available)[0]

'Alcyonidium gelatinosum'

In [11]:
list(names)[0]

'Alcyonidium gelatinosum'

In [17]:
for organism_name, samples in names.items():
    if organism_name not in images_available:
        print(f"{organism_name}\t{samples}")

Reesa vespulae	['SAMEA111458322']
Sciota angustella	['SAMEA111458716', 'SAMEA111458778', 'SAMEA11025012', 'SAMEA11025241', 'SAMEA11025248']
Cerianthus lloydii	['SAMEA14448274', 'SAMEA14452979', 'SAMEA14452980', 'SAMEA14452981', 'SAMEA14452982', 'SAMEA14452983']
Aporrectodea icterica	['SAMEA14448371', 'SAMEA14448375', 'SAMEA14448656', 'SAMEA14448657', 'SAMEA14448658', 'SAMEA14448676', 'SAMEA14448677']
Limecola balthica	['SAMEA112624618', 'SAMEA112652168', 'SAMEA112652171', 'SAMEA112652233', 'SAMEA112652236', 'SAMEA112652244']
Euscorpius flavicaudis	['SAMEA9066036', 'SAMEA9066123', 'SAMEA9066124', 'SAMEA9066125', 'SAMEA9066126', 'SAMEA9066127']
Takobia muticus	['SAMEA9065856', 'SAMEA9065946']
Apocheima pilosaria	['SAMEA9359421', 'SAMEA9359486', 'SAMEA9359490', 'SAMEA9359515', 'SAMEA9359526']


In [13]:
for organism_name in images_available:
    if organism_name not in names:
        print(organism_name)

Synarachnactis lloydii
Takobia mutica
Nephopterix angustella
Tetratrichobothrius flavicaudis
Macoma balthica
Allolobophora icterica
Phigalia pilosaria


In [25]:
data_portal["Phigalia pilosaria"]["images_available"]

True

In [7]:
summary["summary"]["Genome Notes"] = 1183

In [8]:
es.index(index="summary", body=summary["summary"], id="summary")

/var/folders/71/4c_77lrn65x46bxkhf0k6bj80000gp/T/ipykernel_86692/780560005.py:1: DeprecationWarning: The 'body' parameter is deprecated for the 'index' API and will be removed in a future version. Instead use the 'document' parameter. See https://github.com/elastic/elasticsearch-py/issues/1698 for more information
  es.index(index="summary", body=summary["summary"], id="summary")


{'_index': 'summary',
 '_id': 'summary',
 '_version': 14,
 'result': 'updated',
 '_shards': {'total': 2, 'successful': 1, 'failed': 0},
 '_seq_no': 13,
 '_primary_term': 1}

In [17]:
DATA_PORTAL_AGGREGATIONS = ["assemblies_status", "annotation_complete"]

In [22]:
body = dict()
body["aggs"] = dict()
for aggregation_field in DATA_PORTAL_AGGREGATIONS:
    body["aggs"][aggregation_field] = {
        "terms": {"field": aggregation_field, "size": 20}
    }
body["aggs"]["genome_notes"] = {
                "nested": {"path": "genome_notes"},
                "aggs": {
                    "genome_count": {
                        "reverse_nested": {},  # get to the parent document level
                        "aggs": {
                            "distinct_docs": {
                                "cardinality": {
                                    "field": "organism.keyword"
                                }
                            }
                        }
                    }
                }
            }

In [23]:
results = es.search(index="data_portal", body=body)

/var/folders/71/4c_77lrn65x46bxkhf0k6bj80000gp/T/ipykernel_94750/3847356325.py:1: DeprecationWarning: The 'body' parameter is deprecated for the 'search' API and will be removed in a future version. Instead use API parameters directly. See https://github.com/elastic/elasticsearch-py/issues/1698 for more information
  results = es.search(index="data_portal", body=body)


In [26]:
summary = dict()
names_mapping = {
        "assemblies_status": "Assemblies - Submitted",
        "annotation_complete": "Annotation Complete",
        "genome_notes": "Genome Notes"
}
for key, aggs in results["aggregations"].items():
    try:
        for bucket in aggs["buckets"]:
                if bucket['key'] == 'Done':
                    if key in names_mapping:
                        summary[names_mapping[key]] = bucket['doc_count']                
    except KeyError:
        summary["Genome Notes"] = aggs["genome_count"]["distinct_docs"]["value"]

In [25]:
results["aggregations"]

{'assemblies_status': {'doc_count_error_upper_bound': 0,
  'sum_other_doc_count': 0,
  'buckets': [{'key': 'Waiting', 'doc_count': 5559},
   {'key': 'Done', 'doc_count': 1914}]},
 'annotation_complete': {'doc_count_error_upper_bound': 0,
  'sum_other_doc_count': 0,
  'buckets': [{'key': 'Waiting', 'doc_count': 6470},
   {'key': 'Done', 'doc_count': 1003}]},
 'genome_notes': {'doc_count': 1161,
  'genome_count': {'doc_count': 1160, 'distinct_docs': {'value': 1160}}}}

In [27]:
summary

{'Assemblies - Submitted': 1914,
 'Annotation Complete': 1003,
 'Genome Notes': 1160}

In [29]:
es.index(index="summary", body=summary, id="summary")

/var/folders/71/4c_77lrn65x46bxkhf0k6bj80000gp/T/ipykernel_94750/4051931358.py:1: DeprecationWarning: The 'body' parameter is deprecated for the 'index' API and will be removed in a future version. Instead use the 'document' parameter. See https://github.com/elastic/elasticsearch-py/issues/1698 for more information
  es.index(index="summary", body=summary, id="summary")


{'_index': 'summary',
 '_id': 'summary',
 '_version': 1,
 'result': 'created',
 '_shards': {'total': 2, 'successful': 1, 'failed': 0},
 '_seq_no': 0,
 '_primary_term': 1}

In [14]:
data_portal = get_samples("data_portal", es)

In [15]:
data_portal["Salmo trutta"].keys()

dict_keys(['tax_id', 'currentStatus', 'experiment', 'assemblies', 'project_name', 'records', 'taxonomies', 'organism', 'commonName', 'commonNameSource', 'symbionts_experiment', 'symbionts_assemblies', 'symbionts_analyses', 'symbionts_records', 'metagenomes_experiment', 'metagenomes_assemblies', 'metagenomes_records', 'annotation', 'biosamples', 'annotation_status', 'annotation_complete', 'assemblies_status', 'mapped_reads', 'raw_data', 'trackingSystem', 'tolid', 'orgGeoList', 'specGeoList', 'genome_notes', 'goat_info', 'nbnatlas', 'show_tolqc', 'tolqc_links'])

In [16]:
data_portal["Salmo trutta"]["annotation_complete"]

'Done'

In [5]:
data_portal_new["Lemna minuta"]["metagenomes_records"]

[{'accession': 'SAMEA115117662',
  'organism': {'text': 'plant metagenome', 'ontologyTerm': ''},
  'commonName': 'Not specified',
  'sex': None,
  'organismPart': 'WHOLE ORGANISM',
  'tolid': 'laLemMinu1.metagenome',
  'lat': '51.42',
  'lon': '-0.30',
  'locality': 'SURREY | KINGSTON',
  'country': 'United Kingdom',
  'trackingSystem': 'Submitted to BioSamples',
  'lifestage': None,
  'habitat': 'pond'},
 {'accession': 'SAMEA115117663',
  'organism': {'text': 'Armatimonas sp.', 'ontologyTerm': ''},
  'commonName': 'Not specified',
  'sex': None,
  'organismPart': 'WHOLE ORGANISM',
  'tolid': 'laLemMinu1.Armatimonas_sp_1',
  'lat': '51.42',
  'lon': '-0.30',
  'locality': 'SURREY | KINGSTON',
  'country': 'United Kingdom',
  'trackingSystem': 'Submitted to BioSamples',
  'lifestage': None,
  'habitat': 'pond'},
 {'accession': 'SAMEA115117664',
  'organism': {'text': 'Fimbriimonas sp.', 'ontologyTerm': ''},
  'commonName': 'Not specified',
  'sex': None,
  'organismPart': 'WHOLE ORGANIS

In [4]:
data_portal_old = get_samples("2025-04-29_data_portal", es)

In [5]:
for organism_name in data_portal_old:
    if organism_name not in data_portal_new:
        print(organism_name)

Gammarus obtusatus
Andrena carantonica


In [8]:
data_portal_old["Andrena carantonica"]["records"]

[{'accession': 'SAMEA10157803',
  'organism': {'text': 'Andrena carantonica', 'ontologyTerm': ''},
  'commonName': 'Not specified',
  'sex': 'FEMALE',
  'organismPart': 'WHOLE ORGANISM',
  'tolid': 'iyAndCara1',
  'lat': '51.767',
  'lon': '-1.307',
  'locality': 'Berkshire | Wytham woods',
  'country': 'United Kingdom',
  'trackingSystem': 'Submitted to BioSamples',
  'lifestage': 'adult',
  'habitat': 'Woodland'},
 {'accession': 'SAMEA10166736',
  'organism': {'text': 'Andrena carantonica', 'ontologyTerm': ''},
  'commonName': 'Not specified',
  'sex': 'MALE',
  'organismPart': 'WHOLE ORGANISM',
  'tolid': 'iyAndCara2',
  'lat': '51.786',
  'lon': '-1.317',
  'locality': 'Berkshire | Wytham Farm',
  'country': 'United Kingdom',
  'trackingSystem': 'Assemblies - Submitted',
  'lifestage': 'adult',
  'habitat': 'Agricultural land'},
 {'accession': 'SAMEA10200858',
  'organism': {'text': 'Andrena carantonica', 'ontologyTerm': ''},
  'commonName': 'Not specified',
  'sex': 'FEMALE',
  'o

In [6]:
for organism_name in data_portal_new:
    if organism_name not in data_portal_old:
        print(organism_name)

Andrena scotica


In [11]:
def check_raw_data_status(record):
    if 'experiment' in record and len(record['experiment']) > 0:
        return 'Done'
    else:
        return 'Waiting'


def check_assemblies(record):
    if 'assemblies' in record and len(record['assemblies']) > 0:
        return 'Done'
    else:
        return 'Waiting'


def check_annotation_complete(record):
    if record['currentStatus'] == 'Annotation Complete':
        return 'Done'
    else:
        return 'Waiting'

In [12]:
for organism, record in data_portal.items():
    tmp = dict()
    tmp['organism'] = record['organism']
    tmp['commonName'] = record['commonName']
    tmp['biosamples'] = 'Done'
    tmp['biosamples_date'] = None
    tmp['ena_date'] = None
    tmp['annotation_date'] = None
    tmp['raw_data'] = check_raw_data_status(record)
    tmp['mapped_reads'] = tmp['raw_data']
    tmp['assemblies'] = check_assemblies(record)
    tmp['annotation'] = 'Waiting'
    tmp['annotation_complete'] = check_annotation_complete(record)
    tmp['trackingSystem'] = [
        {'name': 'biosamples', 'status': 'Done', 'rank': 1},
        {'name': 'mapped_reads', 'status': tmp['mapped_reads'], 'rank': 2},
        {'name': 'assemblies', 'status': tmp['assemblies'], 'rank': 3},
        {'name': 'raw_data', 'status': tmp['raw_data'], 'rank': 4},
        {'name': 'annotation', 'status': 'Waiting', 'rank': 5},
        {'name': 'annotation_complete', 'status': tmp['annotation_complete'], 'rank': 6}
    ]
    if 'taxonomies' in record:
        tmp['taxonomies'] = record['taxonomies']
    if 'symbionts_records' in record:
        tmp['symbionts_records'] = record['symbionts_records']
    if 'symbionts_assemblies' in record:
        tmp['symbionts_assemblies'] = record['symbionts_assemblies']
    if 'symbionts_status' in record:
        tmp['symbionts_status'] = record['symbionts_status']
    if 'symbionts_experiments' in record:
        tmp['symbionts_experiments'] = record['symbionts_experiments']
    if 'symbionts_biosamples_status' in record:
        tmp['symbionts_biosamples_status'] = record['symbionts_biosamples_status']
    if 'symbionts_assemblies_status' in record:
        tmp['symbionts_assemblies_status'] = record['symbionts_assemblies_status']
    if 'metagenomes_records' in record:
        tmp['metagenomes_records'] = record['metagenomes_records']
    if 'metagenomes_experiments' in record:
        tmp['metagenomes_experiments'] = record['metagenomes_experiments']
    if 'metagenomes_assemblies' in record:
        tmp['metagenomes_assemblies'] = record['metagenomes_assemblies']
    if 'metagenomes_biosamples_status' in record:
        tmp['metagenomes_biosamples_status'] = record['metagenomes_biosamples_status']
    if 'metagenomes_assemblies_status' in record:
        tmp['metagenomes_assemblies_status'] = record['metagenomes_assemblies_status']
    es.index("tracking_status_index", tmp, id=organism)

# Importing Specimens

In [13]:
specimens = list()
with open("/Users/alexey/specimens.jsonl", "r") as f:
    for line in f:
        line = line.rstrip()
        data_record = ast.literal_eval(line)
        specimens.append({"index": {"_index": "organisms_test", "_id": data_record['accession']}})
        specimens.append(data_record)
len(specimens)

110646

In [14]:
es = Elasticsearch([ES_HOST], http_auth=(ES_USERNAME, ES_PASSWORD))
for i in range(0, len(specimens), 5000):
    print(f"Working on {i}: {i+5000}")
    _ = es.bulk(body=specimens[i:i+5000])

Working on 0: 5000
Working on 5000: 10000
Working on 10000: 15000
Working on 15000: 20000
Working on 20000: 25000
Working on 25000: 30000
Working on 30000: 35000
Working on 35000: 40000
Working on 40000: 45000
Working on 45000: 50000
Working on 50000: 55000
Working on 55000: 60000
Working on 60000: 65000
Working on 65000: 70000
Working on 70000: 75000
Working on 75000: 80000
Working on 80000: 85000
Working on 85000: 90000
Working on 90000: 95000
Working on 95000: 100000
Working on 100000: 105000
Working on 105000: 110000
Working on 110000: 115000


# Importing Articles

In [4]:
def get_samples(index_name, es):
    samples = dict()
    data = es.search(index=index_name, size=1000)
    offset = 0
    while len(data['hits']['hits']) > 0:
        for sample in data['hits']['hits']:
            samples[sample['_id']] = sample['_source']
        offset += 1000
        data = es.search(index=index_name, size=1000, from_=offset)
    return samples

In [6]:
es = Elasticsearch([ES_HOST], http_auth=(ES_USERNAME, ES_PASSWORD))
data_portal = get_samples("data_portal", es)

In [7]:
articles = list()
for tax_id, record in data_portal.items():
    print(f"{list(data_portal.keys()).index(tax_id)/len(data_portal)*100}\r", end='', flush=True)
    if 'genome_notes' in record and len(record["genome_notes"]) > 0:
        for article in record["genome_notes"]:
            article_response = requests.get(f"https://www.ebi.ac.uk/europepmc/webservices/rest/search?query={article['study_id']}&format=json").json()
            if len(article_response['resultList']['result']) > 0:
                pub_year = article_response['resultList']['result'][0]['pubYear']
                article['pub_year'] = pub_year
                article['pubYear'] = pub_year
            else:
                article['pub_year'] = None
                article['pubYear'] = None
            article['id'] = article['study_id']
            article['articleType'] = 'Genome Note'
            article['journalTitle'] = 'Wellcome Open Res'
            article['organism_name'] = record['organism']
            articles.append({"index": {"_index": "articles", "_id": article['study_id']}})
            articles.append(article)

99.98493975903614455

In [8]:
len(articles)

2080

In [10]:
articles_old = get_samples("articles", es)

In [12]:
"PRJEB43032" in articles_old

True

In [9]:
data_portal["Eristalis tenax"]["genome_notes"]

[{'tax_id': '198635',
  'study_id': 'PRJEB43008',
  'url': 'https://wellcomeopenresearch.org/articles/6-292/v1',
  'citeURL': 'https://doi.org/10.12688/wellcomeopenres.17267.1',
  'title': 'The genome sequence of the tapered dronefly, <i>Eristalis pertinax</i> (Scopoli, 1763)',
  'abstract': 'We present a genome assembly from an individual male <i>Eristalis tenax </i>(the tapered dronefly; Arthropoda; Insecta; Diptera; Syriphidae). The genome sequence is 487 megabases in span. The majority of the assembly (95.23%) is scaffolded into seven chromosomal pseudomolecules, with the X and Y sex chromosomes assembled. The complete mitochondrial genome was also assembled and is 17.2 kilobases in length.',
  'figureURI': 'https://wellcomeopenresearch.s3.amazonaws.com/manuscripts/19085/7d4a66b1-f0d2-428e-bcea-e64d77da2217_figure1.gif',
  'caption': 'Figure 1. Example image of Eristalis pertinax.',
  'pub_year': '2021',
  'pubYear': '2021',
  'id': 'PRJEB43008',
  'articleType': 'Genome Note',
  '

In [19]:
es = Elasticsearch([ES_HOST], http_auth=(ES_USERNAME, ES_PASSWORD))
for i in range(0, len(articles), 10000):
    print(f"Working on {i}: {i+10000}")
    _ = es.bulk(body=articles[i:i+10000])

Working on 0: 10000


In [21]:
new = get_samples("2025-01-20_data_portal", es)

In [22]:
old = get_samples("2025-01-13_data_portal", es)

In [37]:
new["Eupithecia subfuscata"]['goat_info']['results'][0]['_source']

{'url': 'https://goat.genomehubs.org/records?record_id=216862&result=taxon&taxonomy=ncbi#Eupithecia subfuscata',
 'attributes': [{'name': 'genome_size',
   'value': 640590000,
   'count': 2,
   'aggregation_method': 'median',
   'aggregation_source': 'ancestor'},
  {'name': 'busco_completeness',
   'value': 98.01362088535754,
   'count': 1,
   'aggregation_method': 'max',
   'aggregation_source': 'direct'}]}

In [24]:
old["Eupithecia subfuscata"]['goat_info']

{'url': 'https://goat.genomehubs.org/records?record_id=216862&result=taxon&taxonomy=ncbi#Eupithecia subfuscata',
 'attributes': [{'name': 'genome_size',
   'value': 640590000,
   'count': 2,
   'aggregation_method': 'median',
   'aggregation_source': 'ancestor'},
  {'name': 'busco_completeness',
   'value': 98.01362088535754,
   'count': 1,
   'aggregation_method': 'max',
   'aggregation_source': 'direct'}]}

In [20]:
data_portal['Sisyra nigra'].keys()

dict_keys(['tax_id', 'commonName', 'currentStatus', 'experiment', 'assemblies', 'analyses', 'project_name', 'records', 'taxonomies', 'organism', 'commonNameSource', 'symbionts_experiment', 'symbionts_assemblies', 'symbionts_analyses', 'symbionts_records', 'metagenomes_experiment', 'metagenomes_assemblies', 'metagenomes_analyses', 'metagenomes_records', 'annotation', 'biosamples', 'annotation_status', 'annotation_complete', 'assemblies_status', 'mapped_reads', 'raw_data', 'trackingSystem', 'tolid', 'orgGeoList', 'specGeoList', 'genome_notes', 'goat_info', 'nbnatlas', 'show_tolqc', 'tolqc_links'])

In [21]:
data_portal['Sisyra nigra']['orgGeoList']

[{'organism': 'Sisyra nigra',
  'accession': 'SAMEA112232781',
  'commonName': 'Not specified',
  'sex': 'NOT COLLECTED',
  'organismPart': 'WHOLE ORGANISM',
  'lat': '51.772',
  'lng': '-1.338',
  'locality': 'Berkshire | Wytham Woods'},
 {'organism': 'Sisyra nigra',
  'accession': 'SAMEA7521517',
  'commonName': 'Not specified',
  'sex': 'NOT COLLECTED',
  'organismPart': 'WHOLE ORGANISM',
  'lat': '51.186305',
  'lng': '0.286464',
  'locality': 'England, Kent, Tonbridge'},
 {'organism': 'Sisyra nigra',
  'accession': 'SAMEA7521588',
  'commonName': 'Not specified',
  'sex': 'NOT COLLECTED',
  'organismPart': 'WHOLE ORGANISM',
  'lat': '51.186305',
  'lng': '0.286464',
  'locality': 'England, Kent, Tonbridge'},
 {'organism': 'Sisyra nigra',
  'accession': 'SAMEA112964092',
  'commonName': 'Not specified',
  'sex': 'NOT PROVIDED',
  'organismPart': 'WHOLE ORGANISM',
  'lat': '52.72',
  'lng': '1.48',
  'locality': 'England|Alderfen Broad'},
 {'organism': 'Sisyra nigra',
  'accession':

In [1]:
item = 1
f"test_list{item}" = []

SyntaxError: cannot assign to f-string expression here. Maybe you meant '==' instead of '='? (1536022914.py, line 2)

In [4]:
es = Elasticsearch([ES_HOST], http_auth=(ES_USERNAME, ES_PASSWORD))
def get_samples(index_name, es):
    samples = dict()
    data = es.search(index=index_name, size=1000)
    offset = 0
    while len(data['hits']['hits']) > 0:
        for sample in data['hits']['hits']:
            samples[sample['_id']] = sample['_source']
        offset += 1000
        data = es.search(index=index_name, size=1000, from_=offset)
    return samples

In [1]:
data_portal = get_samples("data_portal", es)

NameError: name 'get_samples' is not defined

In [6]:
tracking_status = get_samples("tracking_status_index", es)

In [10]:
data_portal_new = get_samples("2025-02-06_data_portal", es)

In [14]:
data_portal_new = get_samples("2025-02-06_data_portal", es)

In [15]:
len(data_portal_new)

6640

In [20]:
data_portal_new = get_samples("2025-02-12_data_portal", es)

In [21]:
data_portal_old = get_samples("2025-02-10_data_portal", es)

In [22]:
for organism in data_portal_old:
    if organism not in data_portal_new:
        print(organism)

In [11]:
articles_old = get_samples("articles", es)

In [8]:
len(articles)

1049

In [9]:
articles = list()
for tax_id, record in data_portal.items():
    print(f"{list(data_portal.keys()).index(tax_id)/len(data_portal)*100}\r", end='', flush=True)
    if 'genome_notes' in record and len(record["genome_notes"]) > 0:
        for article in record["genome_notes"]:
            article_response = requests.get(f"https://www.ebi.ac.uk/europepmc/webservices/rest/search?query={article['study_id']}&format=json").json()
            if len(article_response['resultList']['result']) > 0:
                pub_year = article_response['resultList']['result'][0]['pubYear']
                article['pub_year'] = pub_year
                article['pubYear'] = pub_year
            else:
                article['pub_year'] = None
                article['pubYear'] = None
            article['id'] = article['study_id']
            article['articleType'] = 'Genome Note'
            article['journalTitle'] = 'Wellcome Open Res'
            article['organism_name'] = record['organism']
            articles.append(article)

99.98493975903614455

In [10]:
len(articles)

1037

In [14]:
studies = list()
for art in articles:
    studies.append(art["study_id"])

In [15]:
for art_id in articles_old:
    if art_id not in studies:
        print(art_id)

36584110
35100964
36002813
35145030
36395346
36617663
36526530
38491221
37935057
37935059
37738212
PRJEB43008


In [16]:
articles_old["PRJEB43008"]

{'tax_id': '198635',
 'study_id': 'PRJEB43008',
 'url': 'https://wellcomeopenresearch.org/articles/6-292/v1',
 'citeURL': 'https://doi.org/10.12688/wellcomeopenres.17267.1',
 'title': 'The genome sequence of the tapered dronefly, <i>Eristalis pertinax</i> (Scopoli, 1763)',
 'abstract': 'We present a genome assembly from an individual male <i>Eristalis tenax </i>(the tapered dronefly; Arthropoda; Insecta; Diptera; Syriphidae). The genome sequence is 487 megabases in span. The majority of the assembly (95.23%) is scaffolded into seven chromosomal pseudomolecules, with the X and Y sex chromosomes assembled. The complete mitochondrial genome was also assembled and is 17.2 kilobases in length.',
 'figureURI': 'https://wellcomeopenresearch.s3.amazonaws.com/manuscripts/19085/7d4a66b1-f0d2-428e-bcea-e64d77da2217_figure1.gif',
 'caption': 'Figure 1. Example image of Eristalis pertinax.',
 'pub_year': '2021',
 'pubYear': '2021',
 'id': 'PRJEB43008',
 'articleType': 'Genome Note',
 'journalTitle'

In [17]:
es.delete("articles", id="PRJEB43008")

{'_index': 'articles',
 '_id': 'PRJEB43008',
 '_version': 11,
 'result': 'deleted',
 '_shards': {'total': 2, 'successful': 1, 'failed': 0},
 '_seq_no': 13809,
 '_primary_term': 10}

In [7]:
data_portal["Salmo trutta"]["taxonomies"]

{'kingdom': {'scientificName': 'Metazoa',
  'commonName': 'metazoans',
  'tax_id': '33208'},
 'phylum': {'scientificName': 'Chordata',
  'commonName': 'chordates',
  'tax_id': '7711'},
 'class': {'scientificName': 'Actinopteri',
  'commonName': 'Other',
  'tax_id': '186623'},
 'order': {'scientificName': 'Salmoniformes',
  'commonName': 'Other',
  'tax_id': '8006'},
 'family': {'scientificName': 'Salmonidae',
  'commonName': 'salmonids',
  'tax_id': '8015'},
 'genus': {'scientificName': 'Salmo', 'commonName': 'Other', 'tax_id': '8028'},
 'species': {'scientificName': 'Other', 'commonName': 'Other', 'tax_id': None},
 'cohort': {'scientificName': 'Euteleosteomorpha',
  'commonName': 'Other',
  'tax_id': '1489388'},
 'forma': {'scientificName': 'Other', 'commonName': 'Other', 'tax_id': None},
 'infraclass': {'scientificName': 'Teleostei',
  'commonName': 'teleost fishes',
  'tax_id': '32443'},
 'infraorder': {'scientificName': 'Other',
  'commonName': 'Other',
  'tax_id': None},
 'parvord

In [11]:
format(25, 'b').rjust(8, '0') + format(34, 'b').rjust(8, '0')

'0001100100100010'

In [10]:
format(25, 'b')

'11001'

In [13]:
def convertToBinaryString(ip):
        vals = ip.split(".")
        a = format(int(vals[0]), 'b').rjust(8, '0')
        b = format(int(vals[1]), 'b').rjust(8, '0')
        c = format(int(vals[2]), 'b').rjust(8, '0')
        d = format(int(vals[3]), 'b').rjust(8, '0')
        return a+b+c+d

In [17]:
convertToBinaryString("20.0.0.0")[:16] == convertToBinaryString("20.0.12.0")[:16]

True

In [18]:
class Route:
    # A prefix is in form 
    neighbor = ""  # The router that send this router - will be a.b.c.d
    prefix = ""    # The IP address portion of a prefix - will be a.b.c.d
    prefix_len = 0 # The length portion of a prefix - will be an integer
    path = []      # the AS path - list of integers

    def __init__(self, neigh, p, plen, path):
        self.neighbor = neigh
        self.prefix = p
        self.prefix_len = plen
        self.path = path 

    # convert Route to a String    
    def __str__(self):
        return self.prefix+"/"+str(self.prefix_len)+"- ASPATH: " + str(self.path)+", neigh: "+self.neighbor

    # Get the prefix in the a.b.c.d/x format
    def pfx_str(self):
        return self.prefix+"/"+str(self.prefix_len)

In [19]:
r1 = Route("1.1.1.1", "10.0.0.0", 24, [1,2,3])

In [20]:
r2 = Route("1.1.1.1", "10.0.0.0", 24, [1,2,3])

In [22]:
str(r1) == str(r2)

True

In [25]:
convertToBinaryString("10.0.0.0")[:24] in convertToBinaryString("10.0.0.13")

True

In [26]:
convertToBinaryString("10.0.0.0")[:22] in convertToBinaryString("10.0.0.13")

True

In [30]:
a = None

In [31]:
4 < a

TypeError: '<' not supported between instances of 'int' and 'NoneType'

In [34]:
ip, ip_part = "10.0.0.0/24".split("/")

In [37]:
ip, ip_part = str(ip), int(ip_part)

In [43]:
convertToBinaryString("20.0.12.0")[:24] in convertToBinaryString("20.0.12.0")

True

In [42]:
len()

32

In [10]:
!pip install isodate

  Using cached isodate-0.7.2-py3-none-any.whl.metadata (11 kB)

[notice] A new release of pip is available: 24.3.1 -> 26.0
[notice] To update, run: pip install --upgrade pip


In [2]:
  print("=== Example 1: except Exception ===")
  try:
      raise KeyboardInterrupt()
  except Exception as e:
      print(f"Caught by 'except Exception': {type(e).__name__}")  # Never reached
      if isinstance(e, KeyboardInterrupt):
          print("isinstance check hit")  # Never reached
  except KeyboardInterrupt:
      print("KeyboardInterrupt was NOT caught by 'except Exception', fell through to here")

  # Example 2: except BaseException DOES catch KeyboardInterrupt
  print("\n=== Example 2: except BaseException ===")
  try:
      raise KeyboardInterrupt()
  except BaseException as e:
      print(f"Caught by 'except BaseException': {type(e).__name__}")
      if isinstance(e, KeyboardInterrupt):
          print("isinstance check hit — this works with BaseException")

  # Example 3: the pattern from the PR — simulating what actually happens
  print("\n=== Example 3: simulating the PR code ===")
  try:
      try:
          raise KeyboardInterrupt()
      except Exception as e:
          # This block is SKIPPED entirely for KeyboardInterrupt
          print(f"This never prints: {e}")
          raise e
  except KeyboardInterrupt:
      print("KeyboardInterrupt bypassed 'except Exception' completely")

=== Example 1: except Exception ===
KeyboardInterrupt was NOT caught by 'except Exception', fell through to here

=== Example 2: except BaseException ===
Caught by 'except BaseException': KeyboardInterrupt
isinstance check hit — this works with BaseException

=== Example 3: simulating the PR code ===
KeyboardInterrupt bypassed 'except Exception' completely
